<a href="https://colab.research.google.com/github/OSGeoLabBp/tutorials/blob/master/hungarian/pointcloud/pointclouds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Pontfelhők feldolgozása

Ez az összeálltás a gita Műszaki térinformatika egyesület 2026. évi konferenciájának workshopjához készült. A tananyag és a használt képek a BME Általános és Felsőgeodézia Tanszék Geo4All laborjának GitHub oldaláról letölthetők.

## Bevezetés

### Pontfelhőkben tárolt adatok

A pontfelhőkben a pozíció mellett további adatok is tárolásra kerülhetnek.

### Pontfelhők tárolási formátumai

A térinformatikához hasonlóan sokféle adatformátum használatos a pontfelhő állományok tárolására. Ezek között találhatók nyílt formátumok és zárt formátumok.

A nyílt formátumok esetén az állomány belső struktúrájának leírása bárki számára elérhető és ez alapján készíthet olyan programot, mely a formátumot olvasni és írni képes. Néhány, Magyarországon elterjedt nyílt polfelhő formátum: LAS, LAZ, e57, XYZ, PLY, PCD

A zárt formátumok egyes kereskedelmi szotverekhez kötődnek, azokat csak a megfelelő szoftvercsomag megvásárlása esetén tudjuk kezelni. Ilyen formátumok például az RCS és a RCP.

Az egyes tárolási formátumok közötti választásnál mérlegelni kell a metaadatok (intenzitás, szín, osztályozás) megőrzését, a fájlméretet/tömöríthetőséget illetve a szoftver kompatibilitást.

### Pontfelhőlhők előállítási technológiái

A pontfelhők előállításának két fő technológiája létezik a lézeres távmérés (LiDAR) és a fotogrammetria (SfM). A LiDAR technológia több részterületre bontható TLS, ALS, SLAM, stb.

### Nyílt forráskódú programok pontfelhők kezelésére

*   CloudCompare - grafikus felhasználói felület és programozási
*   PDAL - programkönyvtár és parancssori eszközök
*   PCL - programkönyvtár
*   Open3D - programkönyvtár Python programokhoz
*   Potree - internetes megjelenítés



## Nyers pontfelhők előkészítése

A nyers pontfelhők jellemzően zajjal terheltek. Olyan pontok jelenhetnek benne,melyek a valóságban nincsenek illetve a pontok sűrűsége túl nagy az adott feladatunkhoz. Mindkét esetben a pontfelhőben lévő pontok közül bizonyosakat eltávolítunk. A döntési kritériumban van különbség, ami alapján egy pontot megőrzünk vagy kihagyunk.

A zajszűrésnél használt fontosabb módszerek:

* statisztikai módszerek (Statistical Outlier Removal, Radius Outlier Removal)
* geometriai módszerek (pl. RANSAC)
* PCA alapú módszerek
* mélytanulás alapú módszerek




In [2]:
!pip uninstall -q -y ipywidgets
!pip install -q open3d

In [3]:
!wget -q https://github.com/OSGeoLabBp/tutorials/raw/refs/heads/master/hungarian/pointcloud/data/GITA_Test1_Off-ground_points.las
!wget -q https://github.com/OSGeoLabBp/tutorials/raw/refs/heads/master/hungarian/pointcloud/data/minta1.ply

In [4]:
import open3d as o3d

### Statistical Outlier Removal (SOR)

A zaj pontok megtalálása statisztikai alapon. Számítsuk ki minden egyes pontra a **k** legközelebbi szomszéd átlagos távolságát.

$d_i = \frac {1} {k} \sum_{j=1}^k || p_i - p_{ij} ||$

Számítsuk ki ezen távolságok átlagát és szórását

$\mu = \frac 1 {n} \sum_{i=1}^n d_i$

$\sigma = \sqrt \frac {\sum_{i=1}^n (d_i - \mu)^2} {n-1}$

Vizsgáljuk meg minden pontra, hogy a k legközelebbi szomszéd átlagos távolsága nagyobb-e mint a globális átlag szórással megnövelt értéke.

$d_i > \mu + \alpha \cdot \sigma$

$alpha$ általában 1-3 közé eső konstans

In [14]:
NB_NEIGHBORS = 25   # szomszédos pontok száma 10 - 50 között
STD_RATIO = 1       # alfa 1-3 között
pc = o3d.io.read_point_cloud('minta1.ply')
sor_pc, _ = pc.remove_statistical_outlier(nb_neighbors=NB_NEIGHBORS, std_ratio=STD_RATIO)
o3d.io.write_point_cloud('minta1_sor.ply', sor_pc)
print(f"{len(pc.points)-len(sor_pc.points)} pont eltávolítva a {len(pc.points)} közül, {100-len(sor_pc.points)/len(pc.points)*100:.1f}%.")

130018 pont eltávolítva a 1273830 közül, 10.2%.


### Radius Outlier Removal

A kevés közeli szomszéddal rendelkező pontokat tekintjük zajnak. Minden egyes pontra keressük meg egy R sugarú környezetébe eső pontokat, ha ezek száma egy korlátnál kevesebb, akkor eltávolítjuk a pontot.

In [15]:
RADIUS = 0.3
LIMIT = 10
ror_pc, _ = pc.remove_radius_outlier(nb_points=LIMIT, radius=RADIUS)
o3d.io.write_point_cloud('minta1_ror.ply', ror_pc)
print(f"{len(pc.points)-len(ror_pc.points)} pont eltávolítva a {len(pc.points)} közül, {100-len(ror_pc.points)/len(pc.points)*100:.1f}%.")

3376 pont eltávolítva a 1273830 közül, 0.3%.


### Pontfelhő ritkitása

A pontfelhő ritkítására léteznek nagyon egyszerű módszerek. Például minden n. pont eltávolítása vagy véletlenszerűen válaszott pontok eltávolítása. Ezeknél célszerűbb megoldás a szomszédos pontok közötti minimális távolsággal szűrés. Osszuk fel a pontfelhő pontjait a minimális távolság méretű kis kockákra (voxelekre). A voxelen belül csak egy pontot őrizzünk meg.

In [17]:
VOXEL_SIZE = 0.05  # minimális távolság a pontok között
down_pc = pc.voxel_down_sample(voxel_size=VOXEL_SIZE)
print(f"{len(pc.points)-len(down_pc.points)} pont eltávolítva a {len(pc.points)} közül, {100-len(down_pc.points)/len(pc.points)*100:.1f}%.")

597679 pont eltávolítva a 1273830 közül, 46.9%.
